<a href="https://colab.research.google.com/github/lapshinaaa/recsys-tasks/blob/main/DeepRecSys2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep RecSys Course
## Notebook №2

In this notebook, we'll implement various loss functions that are commonly used for training Two-Tower models for CandGen.

### Data
Data are stored in `data.zip`, which consists of:
* `interactions.parquet` - user-item interactions from Yambda dataset (likes for the 500m version)
* `embeddings.parquet` - already filtered and more densely packed embeddings of tracks from Yambda
* `artists.parquet` - items' metadata with mapping into artistsм


In this task, we'll only be interested in `interactions.parquet`

The archive can be downloaded: [from here](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). We're downloading this in the next cell block so there's no need to use the link.

In [1]:
!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -oq dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=623eac40-916e-46f3-b43f-d5a66ab1b7ca
To: /content/dataset.zip
100% 356M/356M [00:06<00:00, 57.9MB/s]


In [3]:
from collections import defaultdict
import copy
import gc
import os
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import polars as pl
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import tests
import math
from tqdm.auto import tqdm

# 0. Data Prep and Metrics

Data processing

In [3]:
# paths to data
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# global vars
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# for reproducibility
np.random.seed(42)

data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################


Metrics

In [4]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################

    """
    Per-user metrics. targets = relevant items G_u, candidates = ranked list R_u (length >= topk).
    Returns hitrate@k(u), recall@k(u), ndcg@k(u).
    """

    recs = candidates[:topk]
    gt = set(targets)

    hits = [1 if item in gt else 0 for item in recs] # faster since gt is a dict
    num_hits = sum(hits)

    # Hitrate@K(u)
    hitrate = 1.0 if num_hits > 0 else 0.0

    # Recall@K(u)
    denom = min(len(targets), topk) # for len can't use len, MUST use original targets
    recall = (num_hits / denom) if denom > 0 else 0.0

    # DCG@K(u)
    dcg = 0.0
    for idx, h in enumerate(hits, start=1):  # idx = 1..K
        if h:
            dcg += 1.0 / math.log2(idx + 1)

    # iDCG@K(u)
    idcg = 0.0
    for idx in range(1, denom + 1):
        idcg += 1.0 / math.log2(idx + 1)

    ndcg = (dcg / idcg) if idcg > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg}


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    """
    Aggregates metrics across users and computes coverage@K.
    Assumes candidates[uid] has length at least topk (or exactly topk, as your note says).
    """
    uids = list(targets.keys())

    hitrate_sum = 0.0
    recall_sum = 0.0
    ndcg_sum = 0.0

    # coverage: union of all recommended items across users in topK
    covered_items = set()

    for uid in uids:
        gt_u = targets[uid]
        rec_u = candidates[uid][:topk]  # guarantee topk slice

        m = get_metrics(gt_u, rec_u, topk=topk)
        hitrate_sum += m["hitrate"]
        recall_sum += m["recall"]
        ndcg_sum += m["ndcg"]

        covered_items.update(rec_u)

    num_users = len(uids)
    hitrate = hitrate_sum / num_users if num_users > 0 else 0.0
    recall = recall_sum / num_users if num_users > 0 else 0.0
    ndcg = ndcg_sum / num_users if num_users > 0 else 0.0

    coverage = (len(covered_items) / catalog_size) if catalog_size > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg, "coverage": coverage}

# 1. Dataset creation and collate func

In this task, you must implement some helper functions to work with user histories of variable length. These functions will later be used when building samplesm batches and when training models, so it is important to ensure a certain format of sequences.

#### Data format: flatten-representation of user history

Instead of storing the history of each user as a separate list (and then do padding to the unified len), we'll store the history of a given batch in one flattened tensor - `flatten` format:

`[u1_t1, u1_t2, ..., u1_tL1, u2_t1, ..., u2_tL2, ...]`

Essentially, in one tensor we'll write interactions of user 1, user 2, etc.

So that we don't lose the boundaries between the users, we'll separately store tensor `length`, where a number of elements of a given history will be stored (for each user in batch):

`length = [L1, L2, ..., LB]`

where `B` — batch size, а `Li` — history len of `i` user.


In this format, we'll store user history (sequence of interactions), but our task is to implement functions that will:
- restore boundaries of sequences using `length`,
- turn flatten-representation into a padded format + mask,
- prepare batch for feeding into a model.

### Function `create_masked_tensor`

Implement function `create_masked_tensor`, which will, using `flatten` representation of the batch of sequences and their lens, form `padded` tensor and bool mask of elements' positions.

First, we'll implement a func that will get lengths and return a tensor with boolean values (`(batch_size, max_len)`).

In [5]:
def get_mask(lengths: torch.Tensor) -> torch.Tensor:
  """
    Creates a boolean mask for variable-length sequences.
  """

 # batch_size = lengths.size(0)
  max_len = lengths.max().item()

  positions = torch.arange(max_len, device=lengths.device)
  mask = positions.unsqueeze(0) < lengths.unsqueeze(1)

  return mask

In [6]:
def create_masked_tensor(data: torch.Tensor, lengths: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
  """
  Converts a batch of flattened variable-length sequences into a padded tensor and mask.
  Supports:
    - indices: data shape (total_num_elements,)
    - embeddings/features: data shape (total_num_elements, d1, d2, ...)

  Parameters
  ----------
  data : torch.Tensor
      Input tensor containing flattened sequences:
      - For indices: shape (total_num_elements,)
      - For embeddings: shape (total_num_elements, embedding_dim)
  lengths : torch.Tensor
      1D tensor of sequence lengths, shape (batch_size,). Specifies the actual length
      of each sequence.

  Returns
  -------
  Tuple[torch.Tensor, torch.Tensor]
      - padded_tensor: Padded tensor of shape:
          - (batch_size, max_seq_len) for indices
          - (batch_size, max_seq_len, embedding_dim) for embeddings
          Shorter sequences are right-padded with zeros.
      - mask: Boolean mask of shape (batch_size, max_seq_len) where True indicates
          valid elements and False indicates padding. Can be used in attention or loss computation.

  Examples
  --------
  >>> data = torch.tensor([1, 2, 3, 4, 5, 6])  # sequences: [1,2], [3,4,5], [6]
  >>> lengths = torch.tensor([2, 3, 1])
  >>> padded, mask = create_masked_tensor(data, lengths)
  >>> padded
  tensor([[1, 2, 0],
          [3, 4, 5],
          [6, 0, 0]])
  >>> mask
  tensor([[ True,  True, False],
          [ True,  True,  True],
          [ True, False, False]])
  """
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################

  mask = get_mask(lengths)
  batch = lengths.size(0)
  elem_max = int(lengths.max().item())

  if data.dim() == 1:
    padded = torch.zeros(batch, elem_max, dtype=data.dtype, device=data.device)
  else:
    D = data.shape[1:]
    padded = torch.zeros(batch, elem_max, *D, dtype=data.dtype, device=data.device)

  start = 0
  for i, length in enumerate(lengths.tolist()):
    end = start + length
    padded[i, :length] = data[start:end]
    start = end

  return padded, mask

In [ ]:
tests.test_create_masked_tensor(create_masked_tensor)

All good! :)


### Class `YambdaDataset`

Implement class `YambdaDataset`, which works with user interaction histories and prepares samples for further model training.

Dataset must support two modes of work, which is flagged by `is_train`, and also cutting the history up to the last `max_seq_len` elements.

In train-mode we turn one user history of length `T` into `T-1` training samples. That is, for a user with history `[i1, i2, ..., iT]` we're creating samples with prefixes:

- `history[:1] -> label = i2`
- `history[:2] -> label = i3`
- ...
- `history[:T-1] -> label = iT`

If we implement it brute force and in `__init__` materialize all such samples, we'll have a lot of data duplicates: the exact same item `i1` will be reappearing in almost all samples, `i2` — in all of them except the first one, etc.

That is why in `__init__` we're storing only indexes/pointers, and the prefix itself of `history` and truncation we're building as we go in `__getitem__`.

#### Inputs

- `histories: Dict[uid, List[int]]` — time-ordered histories of user interactions.
- `labels: Dict[uid, List[int]]` — target items for a user for evaluation (last week in our case).
- `is_train: bool` — dataset mode of work.
- `max_seq_len: int` — max length of a returned history (set to default of `100`).

#### Mode 1: Train mode (`is_train=True`)

In train mode the dataset must prepare samples for user in terms of next-item prediction.

If a user history is: `[i1, i2, ..., iT]`, then `T - 1` samples must be created. For each `t` from `1` to `T-1` (next item position):

- `history` = prefix `history[:t]`, truncated to the last `max_seq_len` elements
- `label` = next item `history[t]`

Format of a train-sample:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  },
  "label": int
}
```

#### Mode 2: Inference mode (`is_train=False`)

In evaluation mode our dataset must return exactly one sample per user.
User is in a dataset only if there are targets for them in `labels`.
Sample contents:
- `history` = user history, truncated to the last `max_seq_len` elements.

Format of an eval-sample:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  }
}
```

In [7]:
class YambdaDataset(Dataset):
  """
  PyTorch Dataset for user interaction histories with next-item prediction samples.

  Parameters
  ----------
  histories : Dict[Any, List[int]]
      Mapping from user id to a list of interacted item ids (sorted by time).
  labels : Dict[Any, List[int]]
      Mapping from user id to a list of target item ids.
      Used only to filter users in eval mode (`uid in labels`).
  is_train : bool
      If True, generate multiple (prefix, next_item) samples per user.
      If False, return one sample per user (filtered by presence in `labels`).
  max_seq_len : int, default 100
      Maximum number of most recent items to keep in the returned history.

  Returns
  -------
  Dict[str, Any]
      Train mode (`is_train=True`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
            "label": int,
          }

      Eval mode (`is_train=False`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
          }

      where:
        - history["item_id"] contains up to `max_seq_len` last items of the selected prefix/history
        - history["length"] is the length of the returned (possibly truncated) history
        - label is a single next item id (int)

  Examples
  --------
  Train mode:
  >>> ds = YambdaDataset(histories, labels={}, is_train=True, max_seq_len=100)
  >>> s = ds[0]
  >>> s["uid"]
  >>> s["history"]["item_id"], s["history"]["length"]
  >>> s["label"]

  Eval mode (filters users by `labels` keys):
  >>> ds = YambdaDataset(histories, labels=test_targets, is_train=False)
  >>> s = ds[0]
  >>> s["uid"]
  >>> s["history"]["item_id"], s["history"]["length"]
  """

  def __init__(
      self,
      histories: Dict[Any, List[int]],
      labels: Dict[Any, List[int]],
      is_train: bool,
      max_seq_len: int = 100,
  ) -> None:
      super().__init__()
      self.histories = histories
      self.labels = labels
      self.is_train = is_train
      self.max_seq_len = max_seq_len
      #####################
      ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
      #####################

      self.samples = []

      if self.is_train:
          for uid, hist in self.histories.items():
              for t in range(1, len(hist)):
                  self.samples.append((uid, t))
      else:
          for uid in self.histories: # we're iterating over keys
              if uid in self.labels:
                  self.samples.append(uid)


  def __len__(self) -> int:
      """Return number of samples (prefix samples in train mode, users in eval mode)."""
      #####################
      ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
      #####################
      return len(self.samples)

  def __getitem__(self, idx: int) -> Dict[str, Any]:
      """
      Build and return a single sample using an index pointer (uid, t).

      In train mode: returns a truncated prefix and the next item as an integer label.
      In eval mode: returns the truncated full history.
      """
      #####################
      ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
      #####################
      if self.is_train:
        uid, t = self.samples[idx]
        full_history = self.histories[uid]

        history = full_history[:t]
        history = history[-self.max_seq_len:]
        label = full_history[t]

        return {
            "uid": uid,
            "history": {
                "item_id": history,
                "length": len(history),
            },
            "label": label,
        }

      else:
        uid = self.samples[idx]
        full_history = self.histories[uid]
        history = full_history[-self.max_seq_len:]

        return {
            "uid": uid,
            "history": {
                "item_id": history,
                "length": len(history),
            },
        }

In [ ]:
tests.test_yambda_dataset(YambdaDataset)

All good! :)


### Function `collate_fn`

Implement the function `collate_fn`, which will be used in `DataLoader` for forming lists of samples
from `YambdaDataset` into batches, that are convenient for passing into the model and working with.

As stated earlier, we're using flatten-representation: instead of padding to the same length we
1) concatinate all user histories from a batch into one 1D-tensor  
2) separately save `length`, so that later we can restore boundaries of sequences


#### Input

`batch: List[Dict[str, Any]]` — list of samples from `YambdaDataset`.

#### What `collate_fn` must do

The function must form a dictionary, where all the elements are `torch.Tensor` of type `torch.long`.

- `result["history"]["item_id"]` 1D tensor, got by concatination all `history["item_id"]` in order of objects in `batch` dim: `(sum(history_lengths),)`

- `result["history"]["length"]` 1D tensor of lengths of histories for each object in batch of dim : `(batch_size,)`

- `result["uid"]` 1D tensor of identifiers of users of dim: `(batch_size,)`

- `result["label"]` (only if input samples contain `"label"`) 1D tensor of lables (next item id) in order of objects in batch dim: `(batch_size,)`

#### Requirements

- Don't use `padding`. Only `flatten`-concatination + `lengths`.
- Store the order of objects in `batch` when concatinating.
- Return `"label"` only it exists in input samples.
- All numeric values must be converted into `torch.Tensor` of type `torch.long`.


In [8]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
  """
  Collate function that converts a list of samples into a **flatten** batch representation.

  This function implements the "flatten" batching scheme: instead of padding variable-length
  sequences to a common length, it concatenates all user histories in the batch into a single
  1D tensor and returns a companion `length` tensor to recover per-user boundaries later.

  The function is compatible with `YambdaDataset` in two modes:
    - Train mode samples contain keys: `"uid"`, `"history"`, and `"label"` (where `"label"` is an `int`).
    - Eval mode samples contain keys: `"uid"` and `"history"`.

  Output batch format
  -------------------
  The returned dictionary contains:
    - `result["history"]["item_id"]`: 1D tensor with all history items concatenated in the
      order of samples in `batch`, shape `(sum(history_lengths),)`, dtype `torch.long`.
    - `result["history"]["length"]`: 1D tensor of per-sample history lengths,
      shape `(batch_size,)`, dtype `torch.long`.
    - `result["uid"]`: 1D tensor of user ids, shape `(batch_size,)`, dtype `torch.long`.
    - If `"label"` is present in the input samples (train batches):
        - `result["label"]`: 1D tensor of labels (next item ids), shape `(batch_size,)`,
          dtype `torch.long`.

  Parameters
  ----------
  batch : List[Dict[str, Any]]
      List of samples returned by the dataset `__getitem__`.

  Returns
  -------
  Dict[str, Any]
      A nested dictionary where all returned values are `torch.Tensor` objects.

  Examples
  --------
  - Train-mode: returns `"history"` + `"uid"` + `"label"` (1D tensor of next-item ids).
  - Eval-mode: returns `"history"` + `"uid"` (no `"label"` key).
  """
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################

  uids=[]
  all_items=[]
  lengths=[]
  labels=[]

  has_label = "label" in batch[0] # flag for train or eval

  for sample in batch: # iterating over each sample (from __getitem__) from our batch (each one is a dict)
    uids.append(sample["uid"])
    all_items.extend(sample["history"]["item_id"])
    lengths.append(sample["history"]["length"])

    if has_label:
      labels.append(sample["label"])

  result = {
      "uid": torch.tensor(uids, dtype=torch.long),
      "history": {
          "item_id": torch.tensor(all_items, dtype=torch.long),
          "length": torch.tensor(lengths, dtype=torch.long),
      }
  }

  if has_label:
    result["label"] = torch.tensor(labels, dtype=torch.long)

  return result

In [ ]:
tests.test_collate_fn(collate_fn)

All good! :)


# 2. Implementation of the computation graph for training a two-tower model (without the loss) and for inference (candidate retrieval).

## UserEncoder

Implement class `UserEncoder`, which is the main component of our model.

`UserEncoder` — is a module, which based on user's interaction history builds their context representation.

Each user $u$ is described by their history of interactions $S_u$.

For each item $i$ from catalog there is a trainable embedding $e_i \in \mathbb{R}^d$.

Representation for a user $u$, $P_u$, is an aggregate of embeddings of all their prior interactions. In this task, as an aggregation bag-of-words-like representation of a user must be implemented based on their history of interactions.

For user $u$ with the history of interactions $i_1, i_2, \ldots, i_{|S_u|}$ , the following must be received:

$$
P_u = \sum_{k=1}^{|S_u|} e_{i_k}.
$$

#### Model input

During training data from `YambdaDataset` using `collate_fn` is transformed into `flatten`-batches and `batch["history"]` is passed as input to the method `UserEncoder.forward` to get user representations from the batch.

#### What the model must do

1. Transform received `item_id` into object embeddings;
2. Calculate representations of user as a tensor of size `(batch_size, embedding_dim)`.
3. Return these representations

In [9]:
class UserEncoder(nn.Module):
  """
  User encoder that represents each user by a cumulative prefix sum of item embeddings.

  Parameters
  ----------
  num_items : int
      Total number of unique items in the catalog.
      Item ids must be in ``[0, num_items - 1]``.
  embedding_dim : int
      Dimension of item embeddings.

  Forward input
  -------------
  inputs : Dict[str, torch.Tensor]
      Dictionary with keys:
      - "item_id": Flattened item indices for concatenated sequences,
        shape ``(total_num_events,)``, dtype ``torch.long``.
      - "length": Per-user sequence lengths, shape ``(batch_size,)``,
        dtype ``torch.long``.

  Forward output
  --------------
  torch.Tensor
      User representations, one vector per user, shape ``(batch_size, embedding_dim)``.
  """
  def __init__(self, num_items: int, embedding_dim: int) -> None:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    super().__init__()
    self.item_embeddings = nn.Embedding(num_items, embedding_dim)

  def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    item_id = inputs["item_id"]
    lengths = inputs["length"]
    embeddings = self.item_embeddings(item_id)



    assert item_id.dtype == torch.long
    assert lengths.dtype == torch.long
    assert item_id.min().item() >= 0
    assert item_id.max().item() < self.item_embeddings.num_embeddings
    assert lengths.sum().item() == item_id.size(0)

    user_representations = []
    start = 0

    for length in lengths.tolist():
      end = start + length
      user_representations.append(embeddings[start:end].sum(dim=0)) # resulting embedding of a user gotten by sum
      start = end

    return torch.stack(user_representations) # just stack all the resulting embeddings of users

In [ ]:
tests.test_user_encoder(UserEncoder)

All good! :)


## TwoTowerModel: training and inference

`TwoTowerModel` combines `UserEncoder` and logic of training/model inference.

#### Notation

$\mathbf{E} \in \mathbb{R}^{|I| \times d}$ — table of item embeddings

$\mathbf{P}_u \in \mathbb{R}^d$ — representation of a user $u$

Relevance of an item $i$ for user $u$: $r_i = \langle \mathbf{E}_{i}, \mathbf{P}_{u}\rangle$.


#### What the model must do

We're given a batch:
`inputs["history"]`: history (based on what we're building a user and training)
`inputs["labels"]`: targets/positives (items from last week based on which we want to get metrics on eval)

For each user in a batch we're:
- building $\mathbf{U}$ based on their history

#### Training mode (`self.training == True`)

- Put `inputs["history"]` through `UserEncoder` and get $\mathbf{U}$ for users in a batch
- calculate loss using method `compute_loss`
- return loss

#### Eval mode (`self.training == False`):

- put `inputs["history"]` through `UserEncoder` and get $\mathbf{U}$ for users in a batch
- calculate: $\text{all\_scores} = \langle\mathbf{U}, \mathbf{E}^{\top}\rangle$ of dim `(batch_size, num_items)`
- return tensor `all_scores` (metrics are calculated separately)


#### Where we're getting postivies from for training

Training is formulated as a task `next item prediction`. For each step in user history next item is considered to be a positive example. In other words, our model is being trained to predict the next object of interaction based on all the previous ones.
    
    

In [10]:
class TwoTower(nn.Module):
  """
  Recommendation model combining user encoder with training and inference logic.

  The model produces:
    - a user representation vector `P_u` via `UserEncoder`
    - an item representation matrix `E` from the embedding table
    - uses dot-product relevance scores: `r_{ui} = <P_u, E_i>`.

  The `forward` method behaves differently depending on `self.training`:

  Training mode (`self.training == True`)
    - Encodes users.
    - Delegates loss computation to `compute_loss(...)`.
    - Returns a loss tensor.

  Evaluation / inference mode (`self.training == False`)
    - Encodes users.
    - Computes scores against all items in the catalog.
    - Returns a full score matrix.

  Parameters
  ----------
  num_items : int
    Total number of unique items in the catalog. Item ids must be in `[0, num_items - 1]`.
  embedding_dim : int
    Dimension of user/item embeddings.

  Notes
  -----
  This base class does not implement `compute_loss`. Subclasses should override it to define a training objective.
  """

  def __init__(self, num_items: int, embedding_dim: int) -> None:
    super().__init__()
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.init_weights(0.02)
    self.item_embeddings = self.encoder.item_embeddings

  @torch.no_grad()
  def init_weights(self, initializer_range: float) -> None:
    """
    Initialize all model parameters with truncated normal distribution.

    Parameters
    ----------
    initializer_range : float
        Standard deviation of the truncated normal initializer.
    """
    for key, value in self.named_parameters():
      assert "weight" in key
      nn.init.trunc_normal_(
        value.data,
        std=initializer_range,
        a=-2 * initializer_range,
        b=2 * initializer_range,
      )

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    """
    Compute training loss.

    Parameters
    ----------
    user_repr : torch.Tensor
        User representations returned by the encoder, shape ``(batch_size, embedding_dim)``.
    inputs : Dict[str, Any]
        Full input batch. Expected to contain at least:
          - ``inputs["history"]``: dict with flattened history fields
          - label information (e.g., ``inputs["label"]``), depending on the training setup

    Returns
    -------
    torch.Tensor
        Scalar loss tensor.
    """
    # This function we will implement later
    # Do not touch nor change it here
    raise NotImplementedError

  def forward(self, inputs: Dict[str, Any]) -> Dict[str, torch.Tensor]:
    """
    Run a forward pass with mode-dependent behavior.
    During training: computes and returns loss.
    During evaluation: computes and returns ranking scores for all items.

    Parameters
    ----------
    inputs : Dict[str, Any]
        Batch dictionary produced by `collate_fn`. Expected keys:
          - ``"history"``: dict with
                - ``"item_id"``: 1D flattened history item ids
                - ``"length"``: per-user history lengths
          - ``"uid"``: user ids tensor

    Returns
    -------
    torch.Tensor
        - If training (self.training == True): loss, scalar tensor
        - If evaluating (self.training == False): all_scores, relevance scores for all items with shape (batch_size, num_items)
    """
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    # “two-tower” idea is that you represent users and items in the same embedding space, and then compare them with a dot product
    users_representation = self.encoder(inputs["history"])

    if self.training == True: # for learning embeddings
      loss = self.compute_loss(user_repr=users_representation, inputs=inputs)
      return loss

    else:
      all_scores = torch.matmul(users_representation, self.item_embeddings.weight.T)
      return all_scores

In [ ]:
tests.test_two_tower(TwoTower)

All good! :)


Taking a look at what we have

In [11]:
data = data.join(
    embeddings.select("item_id"),
    on="item_id",
    how="semi"
)

popular_items = (
    data
    .select("item_id")
    .to_series()
    .value_counts()
    .filter(pl.col("count") >= CORE_MIN_INTERACTIONS_PER_ITEM)
    .select("item_id")
)

data = data.join(popular_items, on="item_id", how="semi")

data = data.join(
    artists,
    on="item_id",
    how="left"
)

item_mapping = (
    data
    .select("item_id")
    .unique()
    .sort("item_id")
    .with_row_index("item_idx")
)

data = data.join(item_mapping, on="item_id", how="left")

max_ts = data.select(pl.col("timestamp").max()).item()
split_ts = max_ts - TEST_INTERVAL_SECONDS

train_df = data.filter(pl.col("timestamp") < split_ts)
test_df  = data.filter(pl.col("timestamp") >= split_ts)

test_df = test_df.join(train_df.select("uid").unique(), on="uid", how="semi")

catalog_size = data["item_idx"].n_unique()

train_histories = dict(
    train_df.group_by("uid")
    .agg(pl.col("item_idx").sort_by("timestamp"))
    .iter_rows()
)

test_targets = dict(
    test_df.group_by("uid")
    .agg(pl.col("item_idx"))
    .iter_rows()
)

In [12]:
TRAIN_BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 2048


yambda_train_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=True
)

yambda_eval_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=False
)

yambda_train_dataloader = DataLoader(
  dataset=yambda_train_dataset,
  batch_size=TRAIN_BATCH_SIZE,
  shuffle=True,
  collate_fn=collate_fn,
  drop_last=True
)

yambda_eval_dataloader = DataLoader(
  dataset=yambda_eval_dataset,
  batch_size=EVAL_BATCH_SIZE,
  shuffle=False,
  collate_fn=collate_fn,
  drop_last=False
)

In [ ]:
print("mapped min/max:", data["item_idx"].min(), data["item_idx"].max())
print("mapped n_unique:", data["item_idx"].n_unique())

mapped min/max: 0 157356
mapped n_unique: 157357


# 3. Train Loop

Implement func `evaluation`, which evaluates the quality of model's recommendations.

The function must:
1) Get top-k recs for each user from `dataloader`.
2) Put them in dict of format `Dict[uid, List[item_id]]`.
3) Calculate metrics by calling `evaluate(...)`, and return the result.

#### Tips & Tricks
- Don't forget to switch the model into `.eval()` mode
- Metrics can be calculated using func `evaluate` from the previous notebook

In [13]:
def evaluation(
  dataloader: DataLoader,
  model: TwoTower,
  catalog_size: int,
  topk: int,
  device: str = "cuda",
) -> Dict[str, float]:
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################
  model.eval()
  candidates={}
  k = min(topk, catalog_size)

  with torch.no_grad():
    for batch in dataloader:
      batch = {
        "uid": batch["uid"].to(device),
        "history": {
          "item_id": batch["history"]["item_id"].to(device),
          "length": batch["history"]["length"].to(device),
        },
      }

      all_scores = model(batch) # (bs, catalog_size)
      topk_items = torch.topk(all_scores, k=k, dim=1).indices # (bs, topk)

      uids = batch["uid"].cpu().tolist()
      item_ids = topk_items.cpu().tolist()

      for uid, item_id in zip(uids, item_ids):
        candidates[uid] = item_id

  metrics = evaluate(
      targets=dataloader.dataset.labels,
      candidates=candidates,
      catalog_size=catalog_size,
      topk=k,
  )

  return metrics

Implement function `train`, which will train our model and then after each epoch evaluation will be performed.

After each epoch the following must be done:
- Run function `evaluation` on `valid_dataloader`
- Display metrics in a readable format.
- Calculate and display average loss on epoch.

Once training is finished, display a message about the end of the training and return the state of the model (`state dict`).

#### Notes

- It's important to correctly change modes of the model:
  - training is performed in `train` mode,
  - validation must be performed inside `evaluation`, where our model is switched into `eval` mode.
- Transferring batch to `device` must work correctly with the structure of the batch, where nested dicts with tensors can occur.

In [14]:
def train(
    train_dataloader: DataLoader,
    valid_dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    catalog_size: int,
    topk: int,
    device: str = "cuda"
) -> Dict[str, Any]:
    model.to(device)

    for epoch in tqdm(range(num_epochs), desc="Training"):
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}", leave=False):
            batch = {
                "uid": batch["uid"].to(device),
                "history": {
                    "item_id": batch["history"]["item_id"].to(device),
                    "length": batch["history"]["length"].to(device),
                },
                "label": batch["label"].to(device),
            }

            optimizer.zero_grad()
            loss = model(batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        average_loss = total_loss / len(train_dataloader)

        metrics = evaluation(
            dataloader=valid_dataloader,
            model=model,
            catalog_size=catalog_size,
            topk=topk,
            device=device,
        )

        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        print(f"Average loss: {average_loss:.4f}")
        print(f"Validation metrics: {metrics}")

    print("\nTraining is complete.")
    return model.state_dict()

# Implement various ways of training our Two-tower model

In [15]:
NUM_EPOCHS = 1
LEARNING_RATE = 1e-3
DEVICE = "cuda"

## 4. Softmax loss



In candidate retrieval task each user and each item have a vector representation of dim $d$ in the shared latent space.

Relevance score of user $u$ and item $i$ is calculated as scalar product of their embeddings:

$$
r(u, i) = \langle \mathbf{E}_i,\;\mathbf{P}_u \rangle,
$$

where:
- $\mathbf{P}_u \in \mathbb{R}^d$ — user representation, gotten from `UserEncoder`;
- $\mathbf{E}_i \in \mathbb{R}^d$ — trainable item representation.

The greater the value $r(u, i)$, the more relevant the item $i$ is for user $u$.

During inference items are sorted in descending order based on their relevance, and the model returns top-K candidates.

In this task, we're training a model on eXtreme Multi-Label Classification.

For each user in batch we need to predict one correct item from the entire catalog of items of size $|\mathcal{I}|$.

#### Formula

Let $i^+$ be the next item for user $u$, then:

$$
\mathcal{L}_{\text{softmax}} = - \sum_{u \in \mathbf{U}} \log p(i^+ \mid u) = - \sum_{u \in \mathbf{U}} \left[r(u, i^+) - \log \sum_{j\in\mathcal{I}} \exp(r(u,j))\right].
$$


#### Why it's the best way to train a model at this stage

Full softmax uses information about the entire catalog: training a model is equal to its usage: we're looking for a positive from the entire catalog on training, we're taking top-K items from the entire catalog on usage (inference).

In [ ]:
class SoftmaxModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    labels = inputs["label"]  # shape: (batch_size,)
    logits = torch.matmul(user_repr, self.item_embeddings.weight.T)  # shape: (batch_size, num_items)
    loss = F.cross_entropy(logits, labels)
    return loss

In [ ]:
tests.test_softmax_model(SoftmaxModel)

All good! :)


In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_full = SoftmaxModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_full = torch.optim.Adam(params=model_full.parameters(), lr=LEARNING_RATE)
best_checkpoint_full  = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_full,
    optimizer=optimizer_full,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1:   0%|          | 0/3747 [00:00<?, ?it/s]


Epoch 1/1
Average loss: 9.9040
Validation metrics: {'hitrate': 0.3252950915985686, 'recall': 0.10273021543313199, 'ndcg': 0.03716004641519007, 'coverage': 0.44674847639444065}

Training is complete.


In [ ]:
model_full.load_state_dict(best_checkpoint_full)
final_metrics_full = evaluation(
    yambda_eval_dataloader,
    model_full,
    catalog_size=catalog_size,
    topk=TOPK
)

In [ ]:
tests.check_softmax_recs(final_metrics_full)

All good! :)


## 5. BCE loss

The main problem of the previous approach - computation and memory: full softmax is usually applicable when a catalog isn't too big - roughly up to tens/hundreds of thousands of items. For catalogs of bigger sizes (millions) simpler approaches are typicallu used. We'll go over them in the order of their complexity. The simplest one of them is Binary Cross-Entropy (BCE).

For each user $u$ we're looking at:

- positive sample: item $i^+$, with which the user really interacted (following user's history);
- negative samples: items $i^-$, sampled from catalog (usually uniformly), with which the user didn't interact.

The model is being trained to predict the probability of an item being relevant to the user at this particular point in time.

#### Formula

For one user $u$, positive item $i^+$ and a set of negative items $\mathcal{I}^-$ loss function looks the following way:

$$
\mathcal{L}_{\text{BCE}} =
- \Big[
\log \sigma\bigl(r(u, i^+)\bigr)
+ \sum_{i^- \in \mathcal{I}^-}
\log \bigl(1 - \sigma(r(u, i^-))\bigr)
\Big]
$$

In [ ]:
class BCEModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    ###
    positives = inputs["label"]
    batch_size = positives.size(0)
    device = positives.device

    # positive scores
    pos_emb = self.item_embeddings(positives)
    pos_logits = (user_repr * pos_emb).sum(dim=1)

    # one negative per user uniformly
    negatives = torch.randint(
        low=0,
        high=self.item_embeddings.num_embeddings,
        size=(batch_size,),
        device=device
    )

    # avoid collision
    mask = negatives == positives
    while mask.any():
        negatives[mask] = torch.randint(
            low=0,
            high=self.item_embeddings.num_embeddings,
            size=(mask.sum().item(),),
            device=device
        )
        mask = negatives == positives

    # negative scores
    neg_emb = self.item_embeddings(negatives)
    neg_logits = (user_repr * neg_emb).sum(dim=1)

    # BCE
    pos_loss = F.binary_cross_entropy_with_logits(
        pos_logits,
        torch.ones_like(pos_logits)
    )
    neg_loss = F.binary_cross_entropy_with_logits(
        neg_logits,
        torch.zeros_like(neg_logits)
    )

    loss = pos_loss + neg_loss
    return loss

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_bce = BCEModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bce = torch.optim.Adam(params=model_bce.parameters(), lr=LEARNING_RATE)
best_checkpoint_bce = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bce,
    optimizer=optimizer_bce,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1:   0%|          | 0/3747 [00:00<?, ?it/s]


Epoch 1/1
Average loss: 0.9100
Validation metrics: {'hitrate': 0.18469262404529188, 'recall': 0.047341554041085046, 'ndcg': 0.016457186695711352, 'coverage': 0.23471469334062037}

Training is complete.


In [ ]:
model_bce.load_state_dict(best_checkpoint_bce)
final_metrics_bce = evaluation(
    yambda_eval_dataloader,
    model_bce,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bce_recs(final_metrics_bce)

All good! :)


## 6. BPR loss

Apart from BCE, Two-tower models can also be trained with Bayesian Personalized Ranking (BPR).

A model is trained on pairs of items:

- positive item $i^+$, that the user actually interacted with;
- negative item $i^-$, that the user didn't interact with (in this approach, chosen uniformly from the catalog).

The goal of training - make sure that for each user the following is true:

$$
r(u, i^+) > r(u, i^-)
$$

Thus, BPR directly optimizes ranking metrics (Recall@K, nDCG@K), making it suitable for retrieval models.

#### Formula

For each user $u$ one positive item $i^+$ and one negative item $i^-$ are chosen. Loss functioin BPR is defined as follows:

$$
\mathcal{L}_{\text{BPR}}
= - \sum_{u \in \mathbf{U}} \log \sigma \bigl(r(u, i^+) - r(u, i^-)\bigr),
$$

where:
- $\mathbf{U}$ - set of users in a batch;
- $r(u, i)$ — relevance score for user $u$ and item $i$;
- $\sigma(x) = \frac{1}{1 + e^{-x}}$ — sigmoid.

**Important note**

BCE asks:
`“Is this item relevant or not?”`

BPR asks:
`“Is the positive item ranked above the negative item?”`

In [ ]:
class BPRModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    positives = inputs["label"]
    batch_size = positives.size(0)
    device = positives.device

    # positive scores
    pos_emb = self.item_embeddings(positives)
    pos_scores = (user_repr * pos_emb).sum(dim=1)

    negatives = torch.randint(
        low=0,
        high=self.item_embeddings.num_embeddings,
        size=(batch_size,),
        device=device
    )

    # avoid collision
    mask = negatives == positives
    while mask.any():
        negatives[mask] = torch.randint(
            low=0,
            high=self.item_embeddings.num_embeddings,
            size=(mask.sum().item(),),
            device=device
        )
        mask = negatives == positives

    # negative scores
    neg_emb = self.item_embeddings(negatives)
    neg_scores = (user_repr * neg_emb).sum(dim=1)

    loss = -F.logsigmoid(pos_scores - neg_scores).mean()
    return loss

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_bpr = BPRModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bpr = torch.optim.Adam(params=model_bpr.parameters(), lr=LEARNING_RATE)
best_checkpoint_bpr = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bpr,
    optimizer=optimizer_bpr,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1:   0%|          | 0/3747 [00:00<?, ?it/s]


Epoch 1/1
Average loss: 0.2517
Validation metrics: {'hitrate': 0.2268867168722961, 'recall': 0.06210333586781143, 'ndcg': 0.02176541964450406, 'coverage': 0.1421290441480201}

Training is complete.


In [ ]:
model_bpr.load_state_dict(best_checkpoint_bpr)
final_metrics_bpr = evaluation(
    yambda_eval_dataloader,
    model_bpr,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bpr_recs(final_metrics_bpr)

All good! :)


## 7. Sampled softmax, uniform negatives

Sampled softmax — is an approximation of full softmax: instead of all items we're taking a small subset thereof and calculate softmax only on it.

For each user $u$ we have:
- positive item $i^+$;
- a set of sampled negatives $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$.

#### Formula

$$
\mathcal{L}_{\text{sampled-uniform}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

In [ ]:
class SampledUniformModel(TwoTower):
  def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
  #  self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
  #  self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    positives = inputs["label"]
    batch_size = positives.size(0)
    device = positives.device
    K = self.num_negatives

    pos_emb = self.item_embeddings(positives)
    pos_scores = (user_repr * pos_emb).sum(dim=1, keepdim=True)

    negatives = torch.randint(
        low=0,
        high=self.item_embeddings.num_embeddings,
        size=(batch_size, K),
        device=device
    )

    mask = negatives == positives.unsqueeze(1)
    while mask.any():
        negatives[mask] = torch.randint(
            low=0,
            high=self.item_embeddings.num_embeddings,
            size=(mask.sum().item(),),
            device=device
        )
        mask = negatives == positives.unsqueeze(1)

    neg_emb = self.item_embeddings(negatives)
    neg_scores = (user_repr.unsqueeze(1) * neg_emb).sum(dim=2)

    logits = torch.cat([pos_scores, neg_scores], dim=1)
    targets = torch.zeros(batch_size, dtype=torch.long, device=device)

    loss = F.cross_entropy(logits, targets)
    return loss

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_uniform = SampledUniformModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_uniform = torch.optim.Adam(params=model_sampled_uniform.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_uniform = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_uniform,
    optimizer=optimizer_sampled_uniform,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1:   0%|          | 0/3747 [00:00<?, ?it/s]


Epoch 1/1
Average loss: 5.5653
Validation metrics: {'hitrate': 0.3241734764727875, 'recall': 0.10206683293816812, 'ndcg': 0.03718482732590991, 'coverage': 0.4361865058434007}

Training is complete.


In [ ]:
model_sampled_uniform.load_state_dict(best_checkpoint_sampled_uniform)
final_metrics_sampled_uniform = evaluation(
    yambda_eval_dataloader,
    model_sampled_uniform,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_uniform_recs(final_metrics_sampled_uniform)

All good! :)


## 8. Sampled softmax, in-batch negatives

In the previous task, we approximated full softmax, sampling negatives uniformly from the catalog. Now, we're going to look into an even more popular approach: in-batch negatives sampling.

For each user $u$ we have:
- positive item $i^+$;
- set of sampled negatives $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$ (but now we're sampling not from the entire catalog, but from a batch).

#### Formula

$$
\mathcal{L}_{\text{sampled-batch}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

In [ ]:
class SampledInBatchModel(TwoTower):
  def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    positives = inputs["label"]
    batch_size = positives.size(0)
    device = positives.device

    pos_emb = self.item_embeddings(positives)
    logits = torch.matmul(user_repr, pos_emb.T)
    targets = torch.arange(batch_size, device=device)

    loss = F.cross_entropy(logits, targets)
    return loss

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_in_batch = SampledInBatchModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_in_batch = torch.optim.Adam(params=model_sampled_in_batch.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch,
    optimizer=optimizer_sampled_in_batch,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1:   0%|          | 0/3747 [00:00<?, ?it/s]


Epoch 1/1
Average loss: 6.3508
Validation metrics: {'hitrate': 0.23326924104043156, 'recall': 0.06408253401885239, 'ndcg': 0.023645719197918382, 'coverage': 0.6724962982263262}

Training is complete.


In [ ]:
model_sampled_in_batch.load_state_dict(best_checkpoint_sampled_in_batch)
final_metrics_sampled_in_batch = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_recs(final_metrics_sampled_in_batch)

## 9. Sampled softmax, in-batch negatives + logq correction

Using in-batch approach is fast and efficient, but there remains one important problem: such distribution of negatives doesn't match the distribution in case of full or uniform sampled softmax.

One of the standard ways to battle the shift - adding `log-q` correction.

#### Why correction is needed

Regular in-batch approach perceives all negatives as "equal", but in reality some items are way more popular than the others that barely appear.

In other words, negatives are like a sample from a certain distribution $q(i)$, and not uniform. If we want to approximate the full softmax, we need to compensate the mentioned shift.

#### Formula

Let $q(i)$ — be the probability of item $i$ appearing as a negative-candidate.

Then we'll correct the logit of negatives:

$$
\tilde{r}(u,i) = r(u,i) - \log q(i),
$$

where $q(i)$ — probability of item $i$ appearing as a negative: frequency of its appearing among positives: $\frac{\#i}{\#all}$.

In [16]:
def build_q_from_train_interactions(
  train_data: pl.DataFrame,
  catalog_size: int,
  item_col: str = "item_id",
  eps: float = 1e-12,
) -> torch.Tensor:
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################
  item_ids = torch.tensor(train_data[item_col].to_list(), dtype=torch.long)
  counts = torch.bincount(item_ids, minlength=catalog_size).float()

  q = counts / counts.sum()
  q = q.clamp_min(eps)
  q = q / q.sum()
  return q

In [ ]:
class SampledInBatchModelLogQ(TwoTower):
  def __init__(
    self,
    num_items: int,
    embedding_dim: int,
    num_negatives: int,
    q: torch.Tensor,
    eps: float = 1e-12,
  ) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.eps = eps

    q = q.detach().float()
    q = q / (q.sum() + eps)
    logq = torch.log(q.clamp_min(eps))
    self.register_buffer("logq", logq)

    self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    positives = inputs["label"]
    batch_size = positives.size(0)
    device = positives.device

    pos_emb = self.item_embeddings(positives) # (B, d) - positive embeddings
    logits = torch.matmul(user_repr, pos_emb.T) # (B, d) @ (d, B) = (B, B)
    batch_logq = self.logq[positives] # (B, )

    logits = logits - batch_logq.unsqueeze(0) # applying same values per column
    targets = torch.arange(batch_size, device=device) # diagonal -> 0, 1, 2, ...

    loss = F.cross_entropy(logits, targets)
    return loss

In [ ]:
gc.collect()
torch.cuda.empty_cache()

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################
q = build_q_from_train_interactions(train_df, catalog_size=catalog_size)
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

model_sampled_in_batch_logq = SampledInBatchModelLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq = torch.optim.Adam(params=model_sampled_in_batch_logq.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq,
    optimizer=optimizer_sampled_in_batch_logq,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1:   0%|          | 0/3747 [00:00<?, ?it/s]


Epoch 1/1
Average loss: 6.4827
Validation metrics: {'hitrate': 0.06203599850451316, 'recall': 0.010379061962489517, 'ndcg': 0.004970492758990075, 'coverage': 0.07305680713282536}

Training is complete.


In [ ]:
model_sampled_in_batch_logq.load_state_dict(best_checkpoint_sampled_in_batch_logq)
final_metrics_sampled_in_batch_logq = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_recs(final_metrics_sampled_in_batch_logq)

# Leaderboards and results

Table with all methods and results on metrics

In [ ]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled softmax, uniform negatives", **final_metrics_sampled_uniform},
    {"method": "Sampled softmax, in-batch negatives", **final_metrics_sampled_in_batch},
    {"method": "Sampled softmax, in-batch negatives + logq correction", **final_metrics_sampled_in_batch_logq},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard

## 10. Theoretical questions

1. What's the main problem of using `Full softmax`?
2. Why did `BCE` perform worse than `BPR`?
3. What could potentially be a problem with the `Sampled softmax, uniform` approach?
4. What's the problem of the `in-batch` approach without `logq`-correction?
5. Why, after adding `logq`, did `coverage` drop?

**Answers:**

1. The main problem of full softmax is its computational and memory cost. For each user, the model has to compute scores against the entire item catalog and normalize over all items. When the catalog is large, especially millions of items, this makes training too expensive in time and GPU memory.

2. BPR often performs better than BCE because BPR is a pairwise ranking loss, while BCE is a pointwise classification loss. In retrieval we care about ranking positives above negatives, and BPR optimizes exactly that: it pushes the positive item to score higher than the negative one. BCE only learns to assign high scores to positives and low scores to negatives independently, so it is less directly aligned with ranking metrics like Recall@K or nDCG.

3. The main problem is sampling bias and approximation error. In sampled softmax with uniform negatives, the model compares the positive item only against a small uniformly sampled subset of the catalog, not against all items. This makes training cheaper, but the sampled negatives may be unrepresentative of the real retrieval problem, especially because in practice negatives are not encountered uniformly.

4. In-batch negatives are biased toward frequent items. Without logq correction, popular items are overrepresented as negatives, so the sampled objective is a biased approximation of full softmax. (this item appears often not necessarily because it is truly the best match, but partly because sampling keeps showing it to us — let’s discount that.)

5. Logq reduces the advantage of popular items, but that does not guarantee more diversity. The model may simply shift to a narrower shared pool of corrected high-scoring items, which lowers coverage.

# Bonus tasks

You've already implemented all the main approaches, calculated metrics and drew conclusions about how different loss functions and strategies of negative sampling affect model's quality. As a bonus, you're offered to implement more advanced methods, one of which we discussed during the lecture.

## 11. Sampled softmax, in-batch negatives + **fixed** logq correction

In this bonus task we will implement a more careful variance of `logq correction`, proposed in the article *Correcting the LogQ Correction: Revisiting Sampled Softmax for Large-Scale Retrieval*. The standard `logq`-correction doesn't entirely mitigate the bias which appears because of a non-uniform distribution of objects in a batch.

The key idea here is that in standard output of `logq` the positive item is interpreted as if it was retrieved from the same distribution as the negatives. In practice, that is not the case: the positive item is always present in the given example definitevely and is not a randomly chosen negative. This detail leads to an additional bias.

Implement **fixed logq correction**: the corrected version of logq, which takes into account the fact that the positive item cannot be processed the same way as sampled negatives.

In [17]:
class SampledInBatchModelFixedLogQ(TwoTower):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q: torch.Tensor,
        eps: float = 1e-12,
    ) -> None:
        super().__init__(num_items=num_items, embedding_dim=embedding_dim)
        self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
        self.num_negatives = num_negatives
        self.eps = eps

        q = q.detach().float()
        q = q / q.sum()
        self.register_buffer("q", q)

        self.init_weights(0.02)

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        positives = inputs["label"]                            # (B,)
        batch_size = positives.size(0)
        device = positives.device

        pos_emb = self.item_embeddings(positives)             # (B, d)
        logits = torch.matmul(user_repr, pos_emb.T)           # (B, B)

        batch_logq = torch.log(self.q[positives].clamp_min(self.eps))   # (B,)

        correction = batch_logq.unsqueeze(0).expand(batch_size, batch_size).clone()
        correction.fill_diagonal_(0.0)

        logits = logits - correction

        targets = torch.arange(batch_size, device=device)
        loss = F.cross_entropy(logits, targets)
        return loss

In [1]:
gc.collect()
torch.cuda.empty_cache()

q = build_q_from_train_interactions(train_df, catalog_size=catalog_size)

model_sampled_in_batch_logq_fixed = SampledInBatchModelFixedLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq_fixed = torch.optim.Adam(params=model_sampled_in_batch_logq_fixed.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq_fixed = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq_fixed,
    optimizer=optimizer_sampled_in_batch_logq_fixed,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

NameError: name 'gc' is not defined

In [ ]:
model_sampled_in_batch_logq_fixed.load_state_dict(best_checkpoint_sampled_in_batch_logq_fixed)
final_metrics_sampled_in_batch_logq_fixed = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq_fixed,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_fixed_recs(final_metrics_sampled_in_batch_logq_fixed)

## 12. Improvement of aggregation of user history

In this bonus task, you are asked to independently improve the way user history is aggregated in the model and achieve an additional quality gain. In the baseline solutions, user history is already used to build the user representation, but the aggregation scheme itself may be fairly simple and may not always capture the order, importance, and context of past interactions well enough.

The goal of this task is to achieve an additional improvement of at least 0.01 in absolute nDCG compared to your best solution from the previous parts. In other words, if your best previous result was, for example, `nDCG@K = 0.123`, then to complete this bonus task you need to achieve at least `0.133`.

Important: the graders did not verify this part in advance themselves, so the extra point will be awarded only for code that is actually reproducible in Google Colab on a T4 GPU and produces the same or very close results. Therefore, in your solution it is especially important to:

* fix random seeds
* clearly specify all changes made to the model
* preserve a correct and complete training pipeline
* not omit important cells for data preparation, training, and evaluation

**Linear Recency Weighting**

The baseline user encoder aggregates all past interactions equally. This ignores the fact that recent interactions are often more informative than old ones. We therefore replace simple summation with recency-aware weighted aggregation, where recent items contribute more strongly to the final user representation.

In [4]:
class UserEncoder(nn.Module):
  def __init__(self, num_items: int, embedding_dim: int) -> None:
    super().__init__()
    self.item_embeddings = nn.Embedding(num_items, embedding_dim)

  def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
    item_id = inputs["item_id"]
    lengths = inputs["length"]
    embeddings = self.item_embeddings(item_id)

    user_representations = []
    start = 0

    for length in lengths.tolist():
      end = start + length
      seq_emb = embeddings[start:end]   # (length, embedding_dim)

      weights = torch.arange(1, length + 1, device=seq_emb.device, dtype=seq_emb.dtype)
      weights = weights / weights.sum()  # normalize
      user_repr = (seq_emb * weights.unsqueeze(1)).sum(dim=0)

      user_representations.append(user_repr)
      start = end

    return torch.stack(user_representations, dim=0)

In [ ]:
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################
final_metrics_your_solution = ...
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

## Leaderboard with bonuses

In [ ]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled, uniform", **final_metrics_sampled_uniform},
    {"method": "Sampled, in-batch", **final_metrics_sampled_in_batch},
    {"method": "Sampled, in-batch + logq", **final_metrics_sampled_in_batch_logq},
    {"method": "Sampled, in-batch + fixed logq", **final_metrics_sampled_in_batch_logq_fixed},
    {"method": "Your custom solution", **final_metrics_your_solution},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard